# Evoluzione del medagliere maschile e femminile

Questo notebook amplia l'analisi sul successo femminile confrontando direttamente il **medagliere maschile** e il **medagliere femminile** nelle Olimpiadi estive, nello stesso arco temporale del progetto: **1964-2020**.

L'obiettivo e' duplice:

1. mostrare come cambia nel tempo la distribuzione delle medaglie tra eventi maschili, femminili e misti;
2. verificare se i risultati maschili e femminili hanno relazioni simili o diverse con gli indicatori socio-economici del dataset `data/csv_per_analisi/dataset_olimpiadi_indicatori_ita.csv`.

I grafici sono realizzati con **Altair** e sono pensati per essere usati anche nella relazione o nella presentazione finale.

In [1]:
from pathlib import Path
import re

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## 1. Dataset usati

Per separare il medagliere per genere uso:

- `Olympic_Athlete_Event_Details.csv`, con atleta, evento, sport, medaglia e NOC;
- `Olympic_Athlete_Biography.csv`, usato per ricavare informazioni sugli atleti e un nome paese leggibile;
- `dataset_olimpiadi_indicatori_ita.csv`, che contiene anno, edizione, nazione e indicatori socio-economici World Bank in italiano.

Il notebook cerca automaticamente la root del progetto, quindi puo' essere eseguito dalla root o da una sottocartella del repository.

In [2]:
def find_project_root() -> Path:
    expected = Path("data") / "csv_olimpiadi" / "Olympic_Athlete_Event_Details.csv"
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / expected).exists():
            return base
    raise FileNotFoundError("Non trovo la root del progetto con la cartella data/.")

PROJECT_ROOT = find_project_root()
OLYMPIC_DIR = PROJECT_ROOT / "data" / "csv_olimpiadi"
ANALYSIS_DIR = PROJECT_ROOT / "data" / "csv_per_analisi"

EVENTS_PATH = OLYMPIC_DIR / "Olympic_Athlete_Event_Details.csv"
BIO_PATH = OLYMPIC_DIR / "Olympic_Athlete_Biography.csv"
SOCIO_PATH = ANALYSIS_DIR / "dataset_olimpiadi_indicatori_ita.csv"

print("Project root:", PROJECT_ROOT)
print("Eventi atleta:", EVENTS_PATH.name)
print("Biografie:", BIO_PATH.name)
print("Indicatori socio-economici:", SOCIO_PATH.name)

Project root: C:\Users\smacchiavelli\development\code\master\progettone
Eventi atleta: Olympic_Athlete_Event_Details.csv
Biografie: Olympic_Athlete_Biography.csv
Indicatori socio-economici: dataset_olimpiadi_indicatori_ita.csv


In [3]:
event_cols = [
    "edition", "country_noc", "sport", "event",
    "athlete_id", "medal", "isTeamSport",
]
bio_cols = ["athlete_id", "sex", "country", "country_noc"]

events = pd.read_csv(EVENTS_PATH, usecols=event_cols)
bio = pd.read_csv(BIO_PATH, usecols=bio_cols)
socio = pd.read_csv(SOCIO_PATH, low_memory=False)

events["athlete_id"] = events["athlete_id"].astype("string")
bio["athlete_id"] = bio["athlete_id"].astype("string")

print(f"Righe atleta-evento: {len(events):,}")
print(f"Righe biografie: {len(bio):,}")
print(f"Righe socio-economiche: {len(socio):,}")
display(socio.head(3))

Righe atleta-evento: 316,834
Righe biografie: 155,861
Righe socio-economiche: 3,368


,Edizione_Olimpiadi,ID_Edizione,Anno,Nazione,Codice_NOC,Medaglie_Oro,Medaglie_Argento,Medaglie_Bronzo,Totale_Medaglie,Paese_Ospitante,Citta_Ospitante,Precipitazioni_Medie_mm,Superficie_Totale_km2,Investimenti_Diretti_Esteri_perc_PIL,Densita_Popolazione,Inflazione_Prezzi_Consumo,Spesa_Militare_Valuta_Locale,Spesa_Militare_perc_PIL,Spesa_Pubblica_Consumi_USD,Spesa_Nazionale_Lorda_Valuta_Locale,Interscambio_Commerciale_perc_PIL,Valore_Aggiunto_Industria_perc_PIL,Valore_Aggiunto_Servizi_Valuta_Locale,Valore_Aggiunto_Servizi_perc_PIL,Inflazione_Deflatore_PIL,PIL_Assoluto_USD,PIL_Valuta_Locale,Crescita_PIL_perc_annua,PIL_Pro_Capite_USD,RNL_Valuta_Locale,RNL_Pro_Capite_USD,Iscrizioni_Scuola_Primaria_perc,Tasso_Mortalita_Neonatale,Tasso_Mortalita_Infantile,Aspettativa_di_Vita,Popolazione_Eta_Lavorativa_perc,Crescita_Demografica_perc,Popolazione_Totale,Popolazione_Urbana,Tasso_Urbanizzazione_perc
0,1964 Summer Olympics,16.000,1964.000,United States,USA,36.000,26.000,28.000,90.000,JPN,Tokyo,715.000,9629090.000,NaN,20.362,1.242,51609620413.943,9.078,NaN,NaN,NaN,NaN,NaN,NaN,1.228,586156715517.241,586156715517.241,4.267,3165.352,587035445116.881,3325.000,NaN,NaN,25.200,70.020,59.770,1.545,185035500.000,130812379.750,70.687
1,1964 Summer Olympics,16.000,1964.000,Soviet Union,RUS,30.000,31.000,35.000,96.000,JPN,Tokyo,NaN,17098240.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.975,67.821,63.134,1.111,121921000.000,67675890.750,55.496
2,1964 Summer Olympics,16.000,1964.000,Japan,JPN,16.000,5.000,8.000,29.000,JPN,Tokyo,1668.000,377800.000,NaN,258.966,5.621,192687500000.000,0.946,NaN,NaN,NaN,NaN,NaN,NaN,5.832,61013284649.766,21964782473915.699,9.809,644.431,21763318171630.801,695.000,NaN,NaN,26.700,68.599,65.650,0.946,94526000.000,61194829.250,64.727


## 2. Medagliere per genere: metodo di conteggio

Il file atleta-evento contiene una riga per atleta. Questo significa che negli sport di squadra una medaglia appare molte volte, una per ciascun componente della squadra. Per ricostruire un medagliere coerente:

- negli sport individuali mantengo le righe-medaglia originali;
- negli sport di squadra conto una sola medaglia per `anno + paese + sport + evento + tipo medaglia`;
- classifico ogni evento come `Maschile`, `Femminile` o `Misto/Altro` leggendo il nome dell'evento.

Gli eventi misti vengono tenuti separati: non li attribuisco arbitrariamente ne' al medagliere maschile ne' a quello femminile.

In [4]:
# Filtro Olimpiadi estive 1964-2020.
events["year"] = events["edition"].str.extract(r"(\d{4})", expand=False).astype(int)
events["is_summer"] = events["edition"].str.contains("Summer Olympics", case=False, na=False)
events = events.loc[events["is_summer"] & events["year"].between(1964, 2020)].copy()

# Normalizzo il flag degli sport di squadra.
if events["isTeamSport"].dtype == "bool":
    events["is_team_sport"] = events["isTeamSport"]
else:
    events["is_team_sport"] = events["isTeamSport"].astype("string").str.strip().str.lower().eq("true")

events = events.merge(bio[["athlete_id", "sex"]], on="athlete_id", how="left")
events["sex"] = events["sex"].fillna("Unknown")
events["has_medal"] = events["medal"].notna() & events["medal"].astype("string").str.strip().ne("")

def classify_event_gender(event: str) -> str:
    text = str(event).lower()
    if re.search(r"\bwomen\b|women's|, women", text):
        return "Femminile"
    if re.search(r"\bmen\b|men's|, men", text):
        return "Maschile"
    return "Misto/Altro"

events["event_gender"] = events["event"].apply(classify_event_gender)

print(f"Righe dopo filtro Summer 1964-2020: {len(events):,}")
display(events["event_gender"].value_counts().to_frame("righe_atleta_evento"))

Righe dopo filtro Summer 1964-2020: 179,141


,righe_atleta_evento
event_gender,
Maschile,108600
Femminile,60315
Misto/Altro,10226


In [5]:
medal_rows = events.loc[events["has_medal"]].copy()

# Allineamento dei codici NOC storici al dataset socio-economico.
# Nel dataset degli indicatori la Soviet Union e' codificata come RUS,
# mentre nel file atleta-evento compare come URS. La sostituzione e' sicura
# perche' avviene su anni in cui non esiste una Russia olimpica separata.
NOC_HISTORICAL_TO_SOCIO = {
    "URS": "RUS",
}
medal_rows["country_noc_original"] = medal_rows["country_noc"]
medal_rows["country_noc"] = medal_rows["country_noc"].replace(NOC_HISTORICAL_TO_SOCIO)
team_medals = (
    medal_rows.loc[medal_rows["is_team_sport"]]
    .drop_duplicates(["year", "country_noc", "sport", "event", "medal"])
)
individual_medals = medal_rows.loc[~medal_rows["is_team_sport"]].copy()
medal_units = pd.concat([individual_medals, team_medals], ignore_index=True)

medals_gender = (
    medal_units
    .groupby(["year", "country_noc", "event_gender"], as_index=False)
    .size()
    .rename(columns={"size": "medals"})
    .pivot_table(
        index=["year", "country_noc"],
        columns="event_gender",
        values="medals",
        fill_value=0,
        aggfunc="sum",
    )
    .reset_index()
)

for col in ["Femminile", "Maschile", "Misto/Altro"]:
    if col not in medals_gender.columns:
        medals_gender[col] = 0

medals_gender = medals_gender.rename(columns={
    "Femminile": "medaglie_femminili",
    "Maschile": "medaglie_maschili",
    "Misto/Altro": "medaglie_miste_altro",
})
medals_gender["medaglie_totali_genere"] = (
    medals_gender["medaglie_femminili"]
    + medals_gender["medaglie_maschili"]
    + medals_gender["medaglie_miste_altro"]
)
medals_gender["quota_femminile"] = np.where(
    (medals_gender["medaglie_femminili"] + medals_gender["medaglie_maschili"]) > 0,
    medals_gender["medaglie_femminili"]
    / (medals_gender["medaglie_femminili"] + medals_gender["medaglie_maschili"]),
    np.nan,
)
medals_gender["gap_f_m"] = medals_gender["medaglie_femminili"] - medals_gender["medaglie_maschili"]

print(f"Medaglie/evento dopo deduplica squadre: {len(medal_units):,}")
display(medals_gender.head())

Medaglie/evento dopo deduplica squadre: 11,781


event_gender,year,country_noc,medaglie_femminili,medaglie_maschili,medaglie_miste_altro,medaglie_totali_genere,quota_femminile,gap_f_m
0,1964,ARG,0,0,1,1,NaN,0
1,1964,AUS,7,10,1,18,0.412,-3
2,1964,BAH,0,0,1,1,NaN,0
3,1964,BEL,0,3,0,3,0.000,-3
4,1964,BRA,0,1,0,1,0.000,-1


## 3. Merge con il dataset socio-economico

Il merge avviene su:

- anno olimpico: `Anno` nel dataset socio-economico;
- paese: `Codice_NOC`, coerente con `country_noc` nei dati olimpici.

Uso il dataset socio-economico come base, cosi' restano anche i paesi che in una certa edizione hanno zero medaglie. Questo e' importante per analizzare correlazioni senza limitarsi solo ai vincitori.

In [6]:
socio = socio.copy()
socio["year"] = pd.to_numeric(socio["Anno"], errors="coerce").astype("Int64")
socio["country_noc"] = socio["Codice_NOC"].astype("string")
socio["country"] = socio["Nazione"].astype("string")

socio_summer = socio.loc[
    socio["Edizione_Olimpiadi"].astype("string").str.contains("Summer Olympics", case=False, na=False)
    & socio["year"].between(1964, 2020)
].copy()
socio_summer["year"] = socio_summer["year"].astype(int)

panel = socio_summer.merge(medals_gender, on=["year", "country_noc"], how="left")

medal_cols = [
    "medaglie_femminili", "medaglie_maschili", "medaglie_miste_altro",
    "medaglie_totali_genere", "gap_f_m",
]
for col in medal_cols:
    panel[col] = panel[col].fillna(0)

panel["quota_femminile"] = np.where(
    (panel["medaglie_femminili"] + panel["medaglie_maschili"]) > 0,
    panel["medaglie_femminili"]
    / (panel["medaglie_femminili"] + panel["medaglie_maschili"]),
    np.nan,
)
panel["quota_maschile"] = np.where(
    (panel["medaglie_femminili"] + panel["medaglie_maschili"]) > 0,
    panel["medaglie_maschili"]
    / (panel["medaglie_femminili"] + panel["medaglie_maschili"]),
    np.nan,
)

print(f"Osservazioni paese-anno nel panel: {len(panel):,}")
print(f"Paesi distinti: {panel['country_noc'].nunique():,}")
display(panel[["year", "country", "country_noc", "medaglie_femminili", "medaglie_maschili", "medaglie_miste_altro"]].head())

Osservazioni paese-anno nel panel: 3,368
Paesi distinti: 259


,year,country,country_noc,medaglie_femminili,medaglie_maschili,medaglie_miste_altro
0,1964,United States,USA,24.000,60.000,6.000
1,1964,Soviet Union,RUS,25.000,69.000,2.000
2,1964,Japan,JPN,2.000,27.000,0.000
3,1964,Germany,GER,9.000,33.000,8.000
4,1964,Italy,ITA,1.000,23.000,3.000


## 4. Distribuzione delle medaglie nel tempo per genere

Il grafico seguente mostra la distribuzione assoluta delle medaglie per anno olimpico. La crescita della componente femminile va letta insieme all'espansione del programma olimpico femminile: piu' eventi disponibili significano piu' opportunita' di medaglia.

In [7]:
global_year = (
    panel.groupby("year", as_index=False)
    .agg(
        medaglie_femminili=("medaglie_femminili", "sum"),
        medaglie_maschili=("medaglie_maschili", "sum"),
        medaglie_miste_altro=("medaglie_miste_altro", "sum"),
    )
)
global_year["totale"] = (
    global_year["medaglie_femminili"]
    + global_year["medaglie_maschili"]
    + global_year["medaglie_miste_altro"]
)
global_year["quota_femminile"] = global_year["medaglie_femminili"] / (
    global_year["medaglie_femminili"] + global_year["medaglie_maschili"]
)

global_long = global_year.melt(
    id_vars=["year"],
    value_vars=["medaglie_maschili", "medaglie_femminili", "medaglie_miste_altro"],
    var_name="genere_evento",
    value_name="medaglie",
)
global_long["genere_evento"] = global_long["genere_evento"].map({
    "medaglie_maschili": "Maschile",
    "medaglie_femminili": "Femminile",
    "medaglie_miste_altro": "Misto/Altro",
})

stacked_medals = (
    alt.Chart(global_long)
    .mark_bar()
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y("medaglie:Q", title="Medaglie"),
        color=alt.Color(
            "genere_evento:N",
            title="Genere evento",
            scale=alt.Scale(domain=["Maschile", "Femminile", "Misto/Altro"], range=["#3569a8", "#c7437a", "#8a8a8a"]),
        ),
        tooltip=[
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("genere_evento:N", title="Genere"),
            alt.Tooltip("medaglie:Q", title="Medaglie", format=",.0f"),
        ],
    )
    .properties(width=780, height=400, title="Distribuzione assoluta delle medaglie per genere dell'evento")
)

stacked_medals

alt.Chart(...)

In [8]:
normalized_medals = (
    alt.Chart(global_long)
    .mark_bar()
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y("medaglie:Q", title="Quota", stack="normalize", axis=alt.Axis(format="%")),
        color=alt.Color(
            "genere_evento:N",
            title="Genere evento",
            scale=alt.Scale(domain=["Maschile", "Femminile", "Misto/Altro"], range=["#3569a8", "#c7437a", "#8a8a8a"]),
        ),
        tooltip=[
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("genere_evento:N", title="Genere"),
            alt.Tooltip("medaglie:Q", title="Medaglie", format=",.0f"),
        ],
    )
    .properties(width=780, height=360, title="Composizione percentuale del medagliere per genere")
)

normalized_medals

alt.Chart(...)

## 5. Avvicinamento tra medagliere maschile e femminile

La linea seguente mostra la quota femminile sul totale delle medaglie assegnate in eventi maschili e femminili. Gli eventi misti sono esclusi dal denominatore per mantenere il confronto pulito.

In [9]:
share_chart = (
    alt.Chart(global_year)
    .mark_line(point=True, strokeWidth=2.5, color="#c7437a")
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y("quota_femminile:Q", title="Quota medaglie femminili", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0, 0.6])),
        tooltip=[
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("quota_femminile:Q", title="Quota femminile", format=".1%"),
            alt.Tooltip("medaglie_femminili:Q", title="Medaglie femm.", format=",.0f"),
            alt.Tooltip("medaglie_maschili:Q", title="Medaglie masch.", format=",.0f"),
        ],
    )
    .properties(width=780, height=360, title="Quota femminile nel medagliere olimpico estivo")
)

share_chart

alt.Chart(...)

## 6. Confronto tra paesi nel periodo recente

Dal 2000 in poi il programma olimpico femminile e' piu' ampio e comparabile. Qui confronto i paesi con piu' medaglie totali nel periodo recente, separando contributo maschile e femminile.

In [10]:
recent = panel.loc[panel["year"] >= 2000].copy()
country_recent = (
    recent.groupby(["country_noc", "country"], as_index=False)
    .agg(
        medaglie_femminili=("medaglie_femminili", "sum"),
        medaglie_maschili=("medaglie_maschili", "sum"),
        medaglie_miste_altro=("medaglie_miste_altro", "sum"),
        PIL_Assoluto_USD=("PIL_Assoluto_USD", "mean"),
        PIL_Pro_Capite_USD=("PIL_Pro_Capite_USD", "mean"),
        Popolazione_Totale=("Popolazione_Totale", "mean"),
        Aspettativa_di_Vita=("Aspettativa_di_Vita", "mean"),
        Tasso_Urbanizzazione_perc=("Tasso_Urbanizzazione_perc", "mean"),
    )
)
country_recent["medaglie_totali"] = (
    country_recent["medaglie_femminili"]
    + country_recent["medaglie_maschili"]
    + country_recent["medaglie_miste_altro"]
)
country_recent["quota_femminile"] = np.where(
    (country_recent["medaglie_femminili"] + country_recent["medaglie_maschili"]) > 0,
    country_recent["medaglie_femminili"]
    / (country_recent["medaglie_femminili"] + country_recent["medaglie_maschili"]),
    np.nan,
)

top_recent = country_recent.nlargest(15, "medaglie_totali")
top_long = top_recent.melt(
    id_vars=["country", "country_noc", "medaglie_totali", "quota_femminile"],
    value_vars=["medaglie_maschili", "medaglie_femminili"],
    var_name="genere",
    value_name="medaglie",
)
top_long["genere"] = top_long["genere"].map({
    "medaglie_maschili": "Maschile",
    "medaglie_femminili": "Femminile",
})

country_bars = (
    alt.Chart(top_long)
    .mark_bar()
    .encode(
        y=alt.Y("country:N", sort="-x", title="Paese"),
        x=alt.X("medaglie:Q", title="Medaglie 2000-2020"),
        color=alt.Color("genere:N", title="Genere", scale=alt.Scale(domain=["Maschile", "Femminile"], range=["#3569a8", "#c7437a"])),
        tooltip=[
            alt.Tooltip("country:N", title="Paese"),
            alt.Tooltip("genere:N", title="Genere"),
            alt.Tooltip("medaglie:Q", title="Medaglie", format=",.0f"),
            alt.Tooltip("quota_femminile:Q", title="Quota femminile", format=".1%"),
        ],
    )
    .properties(width=780, height=440, title="Medagliere maschile e femminile nei principali paesi, 2000-2020")
)

country_bars

alt.Chart(...)

In [11]:
scatter_country = country_recent.loc[country_recent["medaglie_totali"] >= 5].copy()

points = alt.Chart(scatter_country).mark_circle(opacity=0.75, stroke="#333", strokeWidth=0.4).encode(
    x=alt.X("medaglie_maschili:Q", title="Medaglie maschili, 2000-2020"),
    y=alt.Y("medaglie_femminili:Q", title="Medaglie femminili, 2000-2020"),
    size=alt.Size("medaglie_totali:Q", title="Medaglie totali", scale=alt.Scale(range=[40, 900])),
    color=alt.Color("quota_femminile:Q", title="Quota femminile", scale=alt.Scale(scheme="redpurple")),
    tooltip=[
        alt.Tooltip("country:N", title="Paese"),
        alt.Tooltip("medaglie_maschili:Q", title="Maschili", format=",.0f"),
        alt.Tooltip("medaglie_femminili:Q", title="Femminili", format=",.0f"),
        alt.Tooltip("medaglie_totali:Q", title="Totali", format=",.0f"),
        alt.Tooltip("quota_femminile:Q", title="Quota femm.", format=".1%"),
    ],
)

max_val = float(scatter_country[["medaglie_maschili", "medaglie_femminili"]].max().max())
diagonal = alt.Chart(pd.DataFrame({"x": [0, max_val], "y": [0, max_val]})).mark_line(
    strokeDash=[5, 5], color="#777"
).encode(x="x:Q", y="y:Q")

country_scatter = (diagonal + points).properties(
    width=600,
    height=520,
    title="Analogie e differenze tra medagliere maschile e femminile per paese",
).interactive()

country_scatter

alt.LayerChart(...)

## 7. Indicatori socio-economici e risultati per genere

Per evitare che pochi paesi con moltissime medaglie dominino completamente le correlazioni, uso come target `log(1 + medaglie)`. In questo modo il confronto resta leggibile anche con distribuzioni molto sbilanciate.

Gli indicatori piu' asimmetrici, come PIL e popolazione, vengono trasformati in logaritmo.

In [12]:
analysis = panel.copy()

numeric_cols = [
    "PIL_Assoluto_USD", "PIL_Pro_Capite_USD", "Popolazione_Totale",
    "Densita_Popolazione", "Tasso_Urbanizzazione_perc", "Aspettativa_di_Vita",
    "Iscrizioni_Scuola_Primaria_perc", "Popolazione_Eta_Lavorativa_perc",
    "Crescita_Demografica_perc", "Crescita_PIL_perc_annua",
    "Spesa_Militare_perc_PIL", "Investimenti_Diretti_Esteri_perc_PIL",
]
for col in numeric_cols:
    analysis[col] = pd.to_numeric(analysis[col], errors="coerce")

analysis["log_PIL_Assoluto"] = np.log1p(analysis["PIL_Assoluto_USD"])
analysis["log_PIL_Pro_Capite"] = np.log1p(analysis["PIL_Pro_Capite_USD"])
analysis["log_Popolazione"] = np.log1p(analysis["Popolazione_Totale"])
analysis["log_medaglie_maschili"] = np.log1p(analysis["medaglie_maschili"])
analysis["log_medaglie_femminili"] = np.log1p(analysis["medaglie_femminili"])
analysis["log_medaglie_totali"] = np.log1p(
    analysis["medaglie_maschili"] + analysis["medaglie_femminili"] + analysis["medaglie_miste_altro"]
)

indicator_cols = [
    "log_PIL_Assoluto", "log_PIL_Pro_Capite", "log_Popolazione",
    "Densita_Popolazione", "Tasso_Urbanizzazione_perc", "Aspettativa_di_Vita",
    "Iscrizioni_Scuola_Primaria_perc", "Popolazione_Eta_Lavorativa_perc",
    "Crescita_Demografica_perc", "Crescita_PIL_perc_annua",
    "Spesa_Militare_perc_PIL", "Investimenti_Diretti_Esteri_perc_PIL",
]
indicator_labels = {
    "log_PIL_Assoluto": "log(PIL assoluto)",
    "log_PIL_Pro_Capite": "log(PIL pro capite)",
    "log_Popolazione": "log(Popolazione)",
    "Densita_Popolazione": "Densita' popolazione",
    "Tasso_Urbanizzazione_perc": "Urbanizzazione %",
    "Aspettativa_di_Vita": "Aspettativa vita",
    "Iscrizioni_Scuola_Primaria_perc": "Iscrizione primaria %",
    "Popolazione_Eta_Lavorativa_perc": "Eta' lavorativa %",
    "Crescita_Demografica_perc": "Crescita demografica %",
    "Crescita_PIL_perc_annua": "Crescita PIL %",
    "Spesa_Militare_perc_PIL": "Spesa militare % PIL",
    "Investimenti_Diretti_Esteri_perc_PIL": "IDE % PIL",
}
target_cols = ["log_medaglie_maschili", "log_medaglie_femminili", "log_medaglie_totali", "quota_femminile"]
target_labels = {
    "log_medaglie_maschili": "log(1+medaglie maschili)",
    "log_medaglie_femminili": "log(1+medaglie femminili)",
    "log_medaglie_totali": "log(1+medaglie totali)",
    "quota_femminile": "Quota femminile",
}

records = []
for indicator in indicator_cols:
    for target in target_cols:
        tmp = analysis[[indicator, target]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(tmp) >= 30 and tmp[indicator].nunique() > 1 and tmp[target].nunique() > 1:
            records.append({
                "indicatore": indicator_labels[indicator],
                "target": target_labels[target],
                "correlazione": tmp[indicator].corr(tmp[target]),
                "n": len(tmp),
            })

corr_df = pd.DataFrame(records)
display(corr_df.sort_values("correlazione", ascending=False).head(10))

,indicatore,target,correlazione,n
2,log(PIL assoluto),log(1+medaglie totali),0.569,2628
0,log(PIL assoluto),log(1+medaglie maschili),0.545,2628
1,log(PIL assoluto),log(1+medaglie femminili),0.534,2628
10,log(Popolazione),log(1+medaglie totali),0.429,3083
8,log(Popolazione),log(1+medaglie maschili),0.418,3083
9,log(Popolazione),log(1+medaglie femminili),0.382,3083
30,Eta' lavorativa %,log(1+medaglie totali),0.343,3090
28,Eta' lavorativa %,log(1+medaglie maschili),0.327,3090
29,Eta' lavorativa %,log(1+medaglie femminili),0.325,3090
3,log(PIL assoluto),Quota femminile,0.321,638


In [13]:
corr_heatmap = (
    alt.Chart(corr_df)
    .mark_rect()
    .encode(
        x=alt.X("target:N", title="Risultato olimpico"),
        y=alt.Y("indicatore:N", title="Indicatore socio-economico"),
        color=alt.Color(
            "correlazione:Q",
            title="r Pearson",
            scale=alt.Scale(scheme="redblue", domain=[-0.5, 0.8]),
        ),
        tooltip=[
            alt.Tooltip("indicatore:N", title="Indicatore"),
            alt.Tooltip("target:N", title="Target"),
            alt.Tooltip("correlazione:Q", title="Correlazione", format=".3f"),
            alt.Tooltip("n:Q", title="Osservazioni", format=",.0f"),
        ],
    )
    .properties(width=760, height=420, title="Correlazioni tra indicatori socio-economici e risultati olimpici per genere")
)

corr_heatmap

alt.Chart(...)

In [14]:
sex_corr = corr_df.loc[
    corr_df["target"].isin(["log(1+medaglie maschili)", "log(1+medaglie femminili)"])
].copy()
sex_corr["target"] = sex_corr["target"].map({
    "log(1+medaglie maschili)": "Maschile",
    "log(1+medaglie femminili)": "Femminile",
})

corr_bars = (
    alt.Chart(sex_corr)
    .mark_bar()
    .encode(
        y=alt.Y("indicatore:N", title="Indicatore", sort="-x"),
        x=alt.X("correlazione:Q", title="Correlazione con log(1+medaglie)"),
        color=alt.Color("target:N", title="Medagliere", scale=alt.Scale(domain=["Maschile", "Femminile"], range=["#3569a8", "#c7437a"])),
        tooltip=[
            alt.Tooltip("indicatore:N", title="Indicatore"),
            alt.Tooltip("target:N", title="Medagliere"),
            alt.Tooltip("correlazione:Q", title="r", format=".3f"),
            alt.Tooltip("n:Q", title="n", format=",.0f"),
        ],
    )
    .properties(width=760, height=420, title="Maschile vs femminile: quali indicatori correlano di piu'?")
)

corr_bars

alt.Chart(...)

## 8. Scatter: PIL, popolazione e medaglie per genere

Gli scatter seguenti mostrano due relazioni classiche del progetto, ma separate per genere: PIL assoluto e popolazione rispetto al numero di medaglie.

In [15]:
scatter_panel = analysis[[
    "year", "country", "country_noc", "log_PIL_Assoluto", "log_Popolazione",
    "medaglie_maschili", "medaglie_femminili",
]].copy()

scatter_long = scatter_panel.melt(
    id_vars=["year", "country", "country_noc", "log_PIL_Assoluto", "log_Popolazione"],
    value_vars=["medaglie_maschili", "medaglie_femminili"],
    var_name="genere",
    value_name="medaglie",
)
scatter_long["genere"] = scatter_long["genere"].map({
    "medaglie_maschili": "Maschile",
    "medaglie_femminili": "Femminile",
})
scatter_long["log_medaglie"] = np.log1p(scatter_long["medaglie"])
scatter_long = scatter_long.dropna(subset=["log_PIL_Assoluto", "log_medaglie"])

pil_scatter = (
    alt.Chart(scatter_long)
    .mark_circle(opacity=0.35, size=45)
    .encode(
        x=alt.X("log_PIL_Assoluto:Q", title="log(PIL assoluto)"),
        y=alt.Y("log_medaglie:Q", title="log(1 + medaglie)"),
        color=alt.Color("genere:N", title="Medagliere", scale=alt.Scale(domain=["Maschile", "Femminile"], range=["#3569a8", "#c7437a"])),
        tooltip=[
            alt.Tooltip("country:N", title="Paese"),
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("genere:N", title="Genere"),
            alt.Tooltip("medaglie:Q", title="Medaglie", format=",.0f"),
            alt.Tooltip("log_PIL_Assoluto:Q", title="log PIL", format=".2f"),
        ],
    )
    .properties(width=760, height=420, title="PIL assoluto e medaglie: confronto maschile/femminile")
    .interactive()
)

pil_scatter

alt.Chart(...)

In [16]:
pop_scatter_data = scatter_long.dropna(subset=["log_Popolazione", "log_medaglie"])

pop_scatter = (
    alt.Chart(pop_scatter_data)
    .mark_circle(opacity=0.35, size=45)
    .encode(
        x=alt.X("log_Popolazione:Q", title="log(Popolazione)"),
        y=alt.Y("log_medaglie:Q", title="log(1 + medaglie)"),
        color=alt.Color("genere:N", title="Medagliere", scale=alt.Scale(domain=["Maschile", "Femminile"], range=["#3569a8", "#c7437a"])),
        tooltip=[
            alt.Tooltip("country:N", title="Paese"),
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("genere:N", title="Genere"),
            alt.Tooltip("medaglie:Q", title="Medaglie", format=",.0f"),
            alt.Tooltip("log_Popolazione:Q", title="log popolazione", format=".2f"),
        ],
    )
    .properties(width=760, height=420, title="Popolazione e medaglie: confronto maschile/femminile")
    .interactive()
)

pop_scatter

alt.Chart(...)

## 9. Sintesi automatica

Questa cella produce un breve riepilogo numerico utile da trasformare in testo per la relazione.

In [17]:
first_year = int(global_year["year"].min())
last_year = int(global_year["year"].max())
first = global_year.loc[global_year["year"] == first_year].iloc[0]
last = global_year.loc[global_year["year"] == last_year].iloc[0]

top_f = country_recent.sort_values("medaglie_femminili", ascending=False).iloc[0]
top_m = country_recent.sort_values("medaglie_maschili", ascending=False).iloc[0]

best_corr_f = sex_corr.loc[sex_corr["target"] == "Femminile"].sort_values("correlazione", ascending=False).iloc[0]
best_corr_m = sex_corr.loc[sex_corr["target"] == "Maschile"].sort_values("correlazione", ascending=False).iloc[0]

print("Sintesi")
print("-------")
print(
    f"Nel {first_year} la quota femminile delle medaglie era {first['quota_femminile']:.1%}; "
    f"nel {last_year} arriva a {last['quota_femminile']:.1%}."
)
print(
    f"Dal 2000 in poi, il primo paese per medaglie femminili e' {top_f['country']} "
    f"({top_f['medaglie_femminili']:.0f}), mentre per medaglie maschili e' {top_m['country']} "
    f"({top_m['medaglie_maschili']:.0f})."
)
print(
    f"L'indicatore piu' correlato alle medaglie femminili e' {best_corr_f['indicatore']} "
    f"(r={best_corr_f['correlazione']:.3f})."
)
print(
    f"L'indicatore piu' correlato alle medaglie maschili e' {best_corr_m['indicatore']} "
    f"(r={best_corr_m['correlazione']:.3f})."
)

Sintesi
-------
Nel 1964 la quota femminile delle medaglie era 21.0%; nel 2020 arriva a 48.3%.
Dal 2000 in poi, il primo paese per medaglie femminili e' United States (317), mentre per medaglie maschili e' United States (300).
L'indicatore piu' correlato alle medaglie femminili e' log(PIL assoluto) (r=0.534).
L'indicatore piu' correlato alle medaglie maschili e' log(PIL assoluto) (r=0.545).


## Conclusioni

Il confronto per genere permette di aggiungere una lettura piu' fine al tema centrale del progetto.

- Il medagliere femminile cresce nel tempo sia in valore assoluto sia come quota del totale.
- I grandi paesi olimpici tendono a essere forti in entrambi i medaglieri, ma l'equilibrio maschile/femminile varia molto tra paesi.
- Le correlazioni con PIL e popolazione restano importanti per entrambi i generi, ma il confronto puo' evidenziare differenze nella forza del legame con sviluppo, urbanizzazione, aspettativa di vita e altri indicatori.
- La quota femminile non va letta solo come risultato sportivo: e' anche una traccia indiretta della capacita' del sistema sportivo di valorizzare una base piu' ampia di atleti.

Limiti:

- il genere dell'evento e' ricavato dal nome dell'evento;
- gli eventi misti sono separati e non redistribuiti tra maschile e femminile;
- le correlazioni sono descrittive e non dimostrano causalita'.


## Esportazione dei grafici Altair per Jekyll

Questa cella esporta tutti i grafici Altair del notebook in formato JSON Vega-Lite dentro `src/stefano/charts`.

I file JSON possono poi essere copiati o serviti dal sito statico Jekyll e richiamati con il tag `vegachart`, ad esempio:

```liquid
{% vegachart /assets/charts/01_medaglie_per_genere_assolute.json %}
```

Nota: il percorso nel tag dipende da dove Jekyll pubblica la cartella dei chart. Qui l'export locale viene fatto nella cartella del progetto `src/stefano/charts`.


In [ ]:

# Export dei grafici Altair come specifiche Vega-Lite JSON.
# Ogni oggetto chart deve essere gia' stato creato dalle celle precedenti.
CHARTS_DIR = PROJECT_ROOT / "src" / "stefano" / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

charts_to_export = {
    "01_medaglie_per_genere_assolute.json": stacked_medals,
    "02_medaglie_per_genere_percentuale.json": normalized_medals,
    "03_quota_femminile_tempo.json": share_chart,
    "04_medagliere_paesi_genere_2000_2020.json": country_bars,
    "05_scatter_paesi_maschile_femminile.json": country_scatter,
    "06_correlazioni_indicatori_heatmap.json": corr_heatmap,
    "07_correlazioni_indicatori_barre.json": corr_bars,
    "08_scatter_pil_medaglie_genere.json": pil_scatter,
    "09_scatter_popolazione_medaglie_genere.json": pop_scatter,
}

exported_files = []
for filename, chart in charts_to_export.items():
    output_path = CHARTS_DIR / filename
    chart.save(output_path)
    exported_files.append({
        "file": filename,
        "path": str(output_path.relative_to(PROJECT_ROOT)),
        "jekyll_example": "{% vegachart /assets/charts/" + filename + " %}",
    })

exported_charts = pd.DataFrame(exported_files)
print(f"Esportati {len(exported_charts)} grafici in: {CHARTS_DIR}")
display(exported_charts)
